# Import from Code

In [1]:
"""Stabilizer-based entanglement entropy tools for BB codes.

Protocol summary (stabilizer-state method):
1) Build BB stabilizer generators from Hx/Hz.
2) Fix logicals to pick a stabilizer state (default: +1 eigenstate of all logical Z).
3) (Optional) create a stim.Tableau from the stabilizers.
4) For a subsystem A, compute S(A) = rank(G_A_bar) - |A_bar| where G_A_bar is the
   stabilizer generator matrix restricted to the qubits in the complement of A.
5) (Optional) Combine entropies of regions to estimate topological entanglement.
"""

from __future__ import annotations

from dataclasses import dataclass
from functools import lru_cache
from typing import Iterable, List, Optional, Sequence, Tuple, Union

import numpy as np
import stim
import ldpc.mod2 as mod2
import sympy as sp

import pytest

from minimal_ann_theory_logicalX import orthogonalize_logical_x_matrix
from minimal_ann_matrix import _monomial_basis, vector_to_poly
from minimal_ann_theory_logicalX import pair_css_logicals_from_polynomials

from copyreg import dispatch_table
import enum
from turtle import st
from bposd.css import css_code
from numpy import block
from bivariate_bicycle_codes import get_BB_Hx_Hz

x, y = sp.symbols("x y")
from BB_stab_tor import *

## Normal entanglement entropy just fail to determine the tor 

# Construct EPR pair to detect the Tor_1

Construct $\rho = |\Psi^{+}\rangle_{L_j  R} \langle \Psi^{+}|_{L_j  R} \otimes I$

## Detect the mutual information

$I(R: A), I(R: B), I(R: AB)$

No $\text{Tor}_1$

Has $\text{Tor}_1$

## Detect the coherent information

$I_c(R \rangle A):= -S(R|A)=S(A)-S(RA)$

## Detect Synergy

$Σ = I(R:AB) − max(I(R:A), I(R:B))$

In [3]:
##################################################
# Toric code
##################################################

print("\nToric code case")
l = 3
m = 3
c_expr = sp.sympify("1 + x")
d_expr = sp.sympify("1 + y")
# Remember the (0,0) term
a_terms = [(0, 0), (1, 0)]
b_terms = [(0, 0), (0, 1)]
poly_P = sp.sympify("1 + x + x**2" )
poly_Q = sp.sympify("1 + y + y**2" )
# poly_P = sp.sympify("1 + x + x**2 + x**3 + x**4 + x**5" )
# poly_Q = sp.sympify("1 + y + y**2 + y**3 + y**4 + y**5" )
standard_polys = [sp.sympify("1")]
logicals_ann_c = [[apply_periodic_boundary(poly_P * expr, l, m), 0] for expr in standard_polys]
logicals_ann_d = [[0, apply_periodic_boundary(poly_Q * expr, l, m)] for expr in standard_polys]

logicals_ann_c_dualX = [[apply_periodic_boundary(poly_Q * expr, l, m), 0] for expr in standard_polys]
logicals_ann_d_dualX = [[0, apply_periodic_boundary(poly_P * expr, l, m)] for expr in standard_polys]

logical_entries, entropy_entries = main_mutual_info(l, m, a_terms, b_terms, logicals_ann_c + logicals_ann_d, logicals_ann_c_dualX + logicals_ann_d_dualX)


Toric code case
BB_stab_tor module loaded.
logical Z[0] = [x**2 + x + 1, 0]
logical Z[1] = [0, y**2 + y + 1]
logical X[0] = (y**2 + y + 1, 0)
logical X[1] = (0, x**2 + x + 1)
orthogonal X dimension: 2
rank_commutation: 2
[[1 0]
 [0 1]]

Logical X/Z pairing (unique):
  X 0 <-> Z 0
  X 1 <-> Z 1

State type 0
<S_Z, S_X, X_{L_j} X_R, Z_{L_j} Z_R>
logical    I(R:A)    I(R:B)    I(R:AB)
index 0         2         0          2
index 1         0         2          2

State type 1
<S_Z, S_X, X_{L_j} X_R>
logical    I(R:A)    I(R:B)    I(R:AB)
index 0         1         0          1
index 1         0         1          1

State type 2
<S_Z, S_X, Z_{L_j} Z_R>
logical    I(R:A)    I(R:B)    I(R:AB)
index 0         1         0          1
index 1         0         1          1

State type 3
<S_Z, S_X, X_{L_j} X_R, \prod_{i \neq j} Z_{L_{i} >
logical    I(R:A)    I(R:B)    I(R:AB)
index 0         1         0          1
index 1         0         1          1

State type 4
<S_Z, S_X, Z_{L_j} Z_R, \prod

## BB code with torsion

In [ ]:
l = 6
m = 6
print(f"\nl={l}, m={m}, tor case")
c_expr = sp.sympify("x**3 + y + y**2")
d_expr = sp.sympify("y**3 + x + x**2")
a_terms = [(3, 0), (0, 1), (0, 2)]
b_terms = [(0, 3), (1, 0), (2, 0)]
poly_P = sp.sympify("x**3*y**4 + x**3*y**3 + x**3*y**2 + x**3*y + y**2 + 1" )
poly_Q = sp.sympify("x**4*y**3 + x**3*y**3 + x**2*y**3 + x**2 + x*y**3 + 1" )
standard_polys = [sp.sympify("1"), sp.sympify("y"), sp.sympify("y**2"), sp.sympify("y**3"), sp.sympify("x"), sp.sympify("x*y")]
logicals_ann_c = [[apply_periodic_boundary(poly_P * expr, l, m), 0] for expr in standard_polys]
logicals_ann_d = [[0, apply_periodic_boundary(poly_Q * expr, l, m)] for expr in standard_polys]
poly_tor1_c_multiplier = sp.sympify("x**4 + x**3 + x*y**2 + x*y + x + y**2")
poly_tor1_d_multiplier = sp.sympify("x**2 + x*y + x + y**3 + y + 1")
standard_polys_tor = [sp.sympify("1"), sp.sympify("x"), sp.sympify("x**2"), sp.sympify("x**3")]
logicals_tor1 = [[apply_periodic_boundary(poly_tor1_c_multiplier * expr, l, m), apply_periodic_boundary(poly_tor1_d_multiplier * expr, l, m)] for expr in standard_polys_tor]


poly_P_dual = sp.sympify("x**5*y**3 + x**4*y**3 + x**2 + x*y**3 + y**3 + 1" )
poly_Q_dual = sp.sympify("x**3*y**5 + x**3*y**4 + x**3*y + x**3 + y**2 + 1" )
standard_polys = [sp.sympify("1"), sp.sympify("y"), sp.sympify("y**2"), sp.sympify("y**3"), sp.sympify("x"), sp.sympify("x*y")]
# standard_polys = [sp.sympify("1"), sp.sympify("y**5"), sp.sympify("y**10"), sp.sympify("y**15"), sp.sympify("x**5"), sp.sympify("x**5*y**5")]
logicals_ann_c_dualX = [[apply_periodic_boundary(poly_P_dual * expr, l, m), 0] for expr in standard_polys]
logicals_ann_d_dualX = [[0, apply_periodic_boundary(poly_Q_dual * expr, l, m)] for expr in standard_polys]
poly_tor1_c_multiplier_dualX = sp.sympify("x**3*y**4 + x**2*y + x + 1")
poly_tor1_d_multiplier_dualX = sp.sympify("x**4*y + x**3*y + x*y**5 + y**5 + y + 1")
standard_polys_tor_dualX = [sp.sympify("1"), sp.sympify("x"), sp.sympify("x**2"), sp.sympify("x**3")]
# standard_polys_tor_dualX = [sp.sympify("1"), sp.sympify("x**5"), sp.sympify("x**10"), sp.sympify("x**15")]
logicals_tor1_dualX = [[apply_periodic_boundary(poly_tor1_c_multiplier_dualX * expr, l, m), apply_periodic_boundary(poly_tor1_d_multiplier_dualX * expr, l, m)] for expr in standard_polys_tor_dualX]

# logicals_all_z = logicals_ann_c + logicals_ann_d + logicals_tor1
# logicals_all_dualX = logicals_ann_c_dualX + logicals_ann_d_dualX + logicals_tor1_dualX

# logicals_all_z = logicals_ann_c + logicals_ann_d[0:2] + logicals_tor1
# logicals_all_dualX = logicals_ann_c_dualX + logicals_ann_d_dualX[0:2] + logicals_tor1_dualX

logicals_all_z = logicals_ann_c + logicals_ann_d[0:2] + logicals_tor1
logicals_all_dualX = logicals_ann_c_dualX + logicals_ann_d_dualX[0:6] + logicals_tor1_dualX


logical_entries, entropy_entries = main_mutual_info(l, m, a_terms, b_terms, logicals_all_z=logicals_all_z, logicals_all_dualX=logicals_all_dualX)


BB_stab_tor module loaded.

l=m=6, tor case
BB_stab_tor module loaded.
logical Z[0] = [x**3*y**4 + x**3*y**3 + x**3*y**2 + x**3*y + y**2 + 1, 0]
logical Z[1] = [x**3*y**5 + x**3*y**4 + x**3*y**3 + x**3*y**2 + y**3 + y, 0]
logical Z[2] = [x**3*y**5 + x**3*y**4 + x**3*y**3 + x**3 + y**4 + y**2, 0]
logical Z[3] = [x**3*y**5 + x**3*y**4 + x**3*y + x**3 + y**5 + y**3, 0]
logical Z[4] = [x**4*y**4 + x**4*y**3 + x**4*y**2 + x**4*y + x*y**2 + x, 0]
logical Z[5] = [x**4*y**5 + x**4*y**4 + x**4*y**3 + x**4*y**2 + x*y**3 + x*y, 0]
logical Z[6] = [0, x**4*y**3 + x**3*y**3 + x**2*y**3 + x**2 + x*y**3 + 1]
logical Z[7] = [0, x**4*y**4 + x**3*y**4 + x**2*y**4 + x**2*y + x*y**4 + y]
logical Z[8] = [x**4 + x**3 + x*y**2 + x*y + x + y**2, x**2 + x*y + x + y**3 + y + 1]
logical Z[9] = [x**5 + x**4 + x**2*y**2 + x**2*y + x**2 + x*y**2, x**3 + x**2*y + x**2 + x*y**3 + x*y + x]
logical Z[10] = [x**5 + x**3*y**2 + x**3*y + x**3 + x**2*y**2 + 1, x**4 + x**3*y + x**3 + x**2*y**3 + x**2*y + x**2]
logical Z[11] 

Several things that are interseting

- The state type6 are introduced according to the classification of Z. Index 0 to 5 are [Ann(f), 0], and index [6, 7] are [0, Ann(g)], index 8 to 11 are $\text{Tor}_1$
- The logical operator X property demonstrated in type 5 is not obviously as it viewed. The logical operator index 8 to 11 lives on the one side from the first glance. However, it shows nontrivial I(R:A)=1 I(R:B)=1 result. That means the corresponding logical X is the $\text{Tor}_2$, which can lives either on the verticle edge or the horizontal edge.

## Directly from CSS code

In [ ]:
l = 6
m = 6
print(f"\nl={l}, m={m}, tor case")
c_expr = sp.sympify("x**3 + y + y**2")
d_expr = sp.sympify("y**3 + x + x**2")
a_terms = [(3, 0), (0, 1), (0, 2)]
b_terms = [(0, 3), (1, 0), (2, 0)]
paired = pair_css_logicals_from_polynomials(
    f_str="x^3 + y + y^2",
    g_str="y^3 + x + x^2",
    l=l,
    m=m,
)

# Polynomial pairs (one-to-one)
logicals_all_z = paired["z_polys"]
logicals_all_dualX = paired["x_polys"]

logical_entries, entropy_entries = main_mutual_info(l, m, a_terms, b_terms, logicals_all_z=logicals_all_z, logicals_all_dualX=logicals_all_dualX)

BB_stab_tor module loaded.

l=m=6, tor case
BB_stab_tor module loaded.
logical Z[0] = (x**4*y**4 + x**4 + x*y**5 + x*y**2 + x*y + x, 0)
logical Z[1] = (x**4*y**5 + x**4*y + x*y**3 + x*y**2 + x*y + x, 0)
logical Z[2] = (x**5*y**2 + x**5 + x**2*y**4 + x**2*y**3 + x**2*y**2 + x**2*y, 0)
logical Z[3] = (x**5*y**3 + x**5*y + x**2*y**5 + x**2*y**4 + x**2*y**3 + x**2*y**2, 0)
logical Z[4] = (x**5*y**4 + x**5 + x**2*y**5 + x**2*y**2 + x**2*y + x**2, 0)
logical Z[5] = (x**5*y**5 + x**5*y + x**2*y**3 + x**2*y**2 + x**2*y + x**2, 0)
logical Z[6] = (x**4*y + x**4 + x**3*y + x*y**3 + x*y + y**3 + y + 1, x*y**2 + x*y + x + y**2 + y + 1)
logical Z[7] = (x**3*y + x**3 + y + 1, x*y**3 + x + y**3 + 1)
logical Z[8] = (x**4 + x**3 + x*y**2 + x*y + x + y**2, x**2 + x*y + x + y**3 + y + 1)
logical Z[9] = (x**5 + x**3*y + x**2*y**2 + x**2*y + x*y + y**3 + y**2 + y + 1, x**2*y + x + y**2)
logical Z[10] = (x**5*y + x**3 + x**2*y**3 + x**2*y**2 + x*y**2, x**2*y**2 + x*y + y**3)
logical Z[11] = (x**4*y + x**3*y 

The direct CSS code construction mix the different classification class

## BB code no Torsion

In [7]:
l = 3
m = 3
print(f"\nl={l}, m={m}, no tor case")
c_expr = sp.sympify("x**3 + y + y**2")
d_expr = sp.sympify("y**3 + x + x**2")
a_terms = [(3, 0), (0, 1), (0, 2)]
b_terms = [(0, 3), (1, 0), (2, 0)]
poly_P = sp.sympify("1+y" )
poly_Q = sp.sympify("1+x" )
standard_polys = [sp.sympify("1"), sp.sympify("x"), sp.sympify("y"), sp.sympify("x*y")]
logicals_ann_c = [[apply_periodic_boundary(poly_P * expr, l, m), 0] for expr in standard_polys]
logicals_ann_d = [[0, apply_periodic_boundary(poly_Q * expr, l, m)] for expr in standard_polys]

poly_P_dual = sp.sympify("1+x**2" )
poly_Q_dual = sp.sympify("1+y**2" )
standard_polys = [sp.sympify("1"), sp.sympify("x**2"), sp.sympify("y**2"), sp.sympify("x**2*y**2")]
logicals_ann_c_dualX = [[apply_periodic_boundary(poly_P_dual * expr, l, m), 0] for expr in standard_polys]
logicals_ann_d_dualX = [[0, apply_periodic_boundary(poly_Q_dual * expr, l, m)] for expr in standard_polys]

# Without orthogonalization
logicals_all_z = logicals_ann_c + logicals_ann_d
logicals_all_dualX = logicals_ann_c_dualX + logicals_ann_d_dualX

logical_entries, entropy_entries = main_mutual_info(l, m, a_terms, b_terms, logicals_all_z=logicals_all_z, logicals_all_dualX=logicals_all_dualX)


l=3, m=3, no tor case
BB_stab_tor module loaded.
logical Z[0] = [y + 1, 0]
logical Z[1] = [x*y + x, 0]
logical Z[2] = [y**2 + y, 0]
logical Z[3] = [x*y**2 + x*y, 0]
logical Z[4] = [0, x + 1]
logical Z[5] = [0, x**2 + x]
logical Z[6] = [0, x*y + y]
logical Z[7] = [0, x**2*y + x*y]
logical X[0] = (x**2 + 1, 0)
logical X[1] = (x**2 + x, 0)
logical X[2] = (x**2*y**2 + y**2, 0)
logical X[3] = (x**2*y**2 + x*y**2, 0)
logical X[4] = (0, y**2 + 1)
logical X[5] = (0, x**2*y**2 + x**2)
logical X[6] = (0, y**2 + y)
logical X[7] = (0, x**2*y**2 + x**2*y)
orthogonal X dimension: 8
rank_commutation: 8
[[1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 1]]

Logical X/Z pairing (unique):
  X 0 <-> Z 0
  X 1 <-> Z 1
  X 2 <-> Z 2
  X 3 <-> Z 3
  X 4 <-> Z 4
  X 5 <-> Z 5
  X 6 <-> Z 6
  X 7 <-> Z 7

State type 0
<S_Z, S_X, X_{L_j} X_R, Z_{L_j} Z_R>
logical    I(R:A)    I(R:B)    I(R:AB)
index 0         2 